## Implemented approach (read this first)

**This notebook runs spectral clustering twice on the same blended title-similarity matrix.** It does not cluster semantic embeddings alone. The shared matrix from Part 3 of `final_report.ipynb` combines semantic similarity, TF-IDF, fuzzy title matching, and the conflict penalty.

The objects being clustered at both levels are the **445 original feature-title strings**:

- A **parent** is a Level 1 cluster label assigned to a set of titles by spectral clustering on the full 445-by-445 matrix. It is not a keyword category, and no title is promoted into a parent.
- A **parent representative** is one actual title (the medoid) printed only as a convenient description of that cluster. It does not determine membership.
- A **child** is a Level 2 cluster of titles assigned by running spectral clustering again on the similarity-matrix slice for one parent. Each child belongs to exactly one parent; every title has one parent and one child.

For example, in the current default run, a parent whose representative is **“Insulating”** means “the titles the algorithm assigned to that parent.” It does **not** mean “all titles containing the word insulating.” Titles without that word can be members, and titles with it can be elsewhere. The printed tree is the source of truth for membership.

## Detailed workflow

1. Load and validate `data/similarity.npz`. The saved `titles` list gives the row/column order, so matrix positions can be mapped back to feature-title strings.
2. Set `K_PARENT` in the settings cell. The notebook builds the full symmetric k-nearest-neighbor graph, computes its normalized-Laplacian eigenvectors, then uses k-means on those eigenvectors to assign every title to exactly one parent label. The default is 8; change it to compare broader or more numerous parent partitions.
3. For each parent label, collect its title indices and slice the original matrix to just those rows and columns. This makes the parent a boundary: Level 2 does not compare or merge titles across different parents.
4. Rebuild the local neighbor graph and repeat normalized spectral clustering within each parent. By default, a local eigengap chooses that parent's child count between `CHILD_K_MIN` and `CHILD_K_MAX`; alternatively, set `K_CHILD_MODE = "fixed"` and choose `K_CHILD_FIXED`.
5. **Children partition their parent.** Spectral clustering assigns every title in a parent to exactly one child, so child sizes must add up to the parent size. For example, a parent of 57 titles split into children of 33, 20, and 4 has no leftover titles: `33 + 20 + 4 = 57`. Unlike DBSCAN, this spectral workflow has no noise/unassigned label.
6. Inspect the printed tree. Each parent heading includes its medoid title, and every child lists its own medoid plus **all titles assigned to that child** by default. Together, the child lists account for every title in the parent. Set `MAX_TREE_MEMBERS` to an integer only if you want to truncate this output.
7. Inspect the single t-SNE plot. It is a display of the full distance matrix, not an input to clustering: marker **color** represents the parent, marker **shape** represents the child number within that parent, and hover shows the title and both assignments.
8. Use the parent-count sweep and metrics as supporting evidence, then judge whether the actual title lists make useful groups. A two-level result adds organization, not new similarity evidence, and a parent boundary can separate titles that are highly similar.

Cluster numbers have no inherent meaning. Use the printed title membership and representatives to decide whether the groups are useful.


In [69]:
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.colors import qualitative
from plotly.subplots import make_subplots
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

# Main hierarchy controls
K_PARENT = 8  # number of Level 1 parent clusters; membership is not keyword-based
K_CHILD_MODE = "eigengap"  # "eigengap" or "fixed"
K_CHILD_FIXED = 2          # used only when K_CHILD_MODE == "fixed"
CHILD_K_MIN = 2
CHILD_K_MAX = 6

# Keep the graph construction and random seed aligned with 5_spectral.ipynb
K_NEIGHBORS = 15
SEED = 0
PARENT_K_SWEEP = [2, 3, 4, 6, 8, 10, 12, 16, 20, 25]
SWEEP_SEEDS = [0, 1, 2, 3, 4]  # repeat K-means to show initialization variability
MAX_TREE_MEMBERS = None  # None prints every title in the hierarchy tree

with np.load("data/similarity.npz") as data:
    S = data["similarity"]
    attributes = data["titles"].tolist()

n = len(attributes)
assert S.shape == (n, n), f"matrix {S.shape} does not match {n} titles"
assert len(set(attributes)) == n, "duplicate title names"
assert S.min() >= 0 and S.max() <= 1 + 1e-9, "similarities must be in [0, 1]"
assert np.allclose(S, S.T), "similarity matrix must be symmetric"
assert 2 <= K_PARENT < n, "K_PARENT must be at least 2 and smaller than the title count"
assert K_CHILD_MODE in {"eigengap", "fixed"}
assert K_CHILD_FIXED >= 2
assert 2 <= CHILD_K_MIN <= CHILD_K_MAX

print(f"Loaded {n} titles and similarity matrix {S.shape}")
print(f"Level 1 parent count is set to K_PARENT={K_PARENT}")


Loaded 445 titles and similarity matrix (445, 445)
Level 1 parent count is set to K_PARENT=8


## 1. Build the affinity graph

Each title is a graph node and similarity is the edge weight. As in the original spectral notebook, retain a link when either endpoint includes the other in its top `K_NEIGHBORS`. The diagonal is removed so a title does not count itself as a neighbor. For the smaller parent graphs at Level 2, the neighbor count is safely capped at `group_size - 1`.


In [70]:
def build_affinity(similarity, k_neighbors=K_NEIGHBORS):
    size = len(similarity)
    if similarity.shape != (size, size):
        raise ValueError("similarity must be square")
    if size < 2:
        return np.zeros_like(similarity)
    if k_neighbors < 1:
        raise ValueError("k_neighbors must be positive")

    k_neighbors = min(k_neighbors, size - 1)
    W = similarity.copy()
    np.fill_diagonal(W, 0)
    threshold = np.sort(W, axis=1)[:, -k_neighbors][:, None]
    W = np.where((W >= threshold) | (W >= threshold.T), W, 0)
    return W


def normalized_laplacian(W):
    degree = W.sum(axis=1)
    inv_sqrt_degree = 1 / np.sqrt(np.maximum(degree, 1e-12))
    return np.eye(len(W)) - W * np.outer(inv_sqrt_degree, inv_sqrt_degree)


def spectral_embedding(W, n_eigen):
    eigenvalues, eigenvectors = np.linalg.eigh(normalized_laplacian(W))
    return eigenvalues[:n_eigen], eigenvectors[:, :n_eigen]


def spectral_cluster(eigenvectors, k, seed=SEED):
    if not 1 <= k <= len(eigenvectors):
        raise ValueError(f"k={k} is invalid for {len(eigenvectors)} items")
    U = eigenvectors[:, :k]
    U = U / np.maximum(np.linalg.norm(U, axis=1, keepdims=True), 1e-12)
    raw_labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(U)
    _, compact_labels = np.unique(raw_labels, return_inverse=True)
    return compact_labels, U


W = build_affinity(S)
assert np.allclose(W, W.T)

degree = W.sum(axis=1)
neighbors = (W > 0).sum(axis=1)
n_components, _ = connected_components(W > 0, directed=False)
possible_edges = n * (n - 1) // 2
kept_edges = int(np.count_nonzero(np.triu(W)))

print(f"Edges kept: {kept_edges} of {possible_edges} ({kept_edges / possible_edges:.1%})")
print(f"Neighbors per title: min {neighbors.min()}, median {int(np.median(neighbors))}, max {neighbors.max()}")
print(f"Weighted degree: min {degree.min():.2f}, median {np.median(degree):.2f}, max {degree.max():.2f}")
print(f"Isolated titles: {(degree == 0).sum()}, connected components: {n_components}")


Edges kept: 4618 of 98790 (4.7%)
Neighbors per title: min 15, median 18, max 59
Weighted degree: min 1.01, median 5.60, max 18.85
Isolated titles: 0, connected components: 1


## 2. Level 1: create parent groups

Unlike the eigengap-only choice in the original notebook, `K_PARENT` is an explicit setting at the top. Change it and rerun from the settings cell to compare broader or more numerous parent groups. The sweep below gives quality and size context, but no metric can decide whether a group is a useful human category; inspect the tree and visualization too.


In [71]:
max_parent_k = max([K_PARENT] + [k for k in PARENT_K_SWEEP if 2 <= k < n])
parent_eigenvalues, parent_eigenvectors = spectral_embedding(W, n_eigen=max_parent_k)
parent_labels, parent_embedding = spectral_cluster(parent_eigenvectors, K_PARENT)
parent_sizes = Counter(parent_labels)

print(f"Parent groups: {len(parent_sizes)}")
print(f"Parent sizes (largest first): {[size for _, size in parent_sizes.most_common()]}")


Parent groups: 8
Parent sizes (largest first): [130, 57, 56, 55, 48, 43, 35, 21]


## 3. Level 2: cluster within each parent

Each parent gets its own similarity submatrix and affinity graph. By default, the child count is selected with a local eigengap between `CHILD_K_MIN` and `CHILD_K_MAX`; set `K_CHILD_MODE = "fixed"` to use `K_CHILD_FIXED` children per parent.

This is a partition, not a filter: spectral clustering assigns every title in a parent to exactly one child. There is no noise class. Therefore, for every parent, the sizes of all its children add up to the parent size. A parent of 57 titles with child sizes 33, 20, and 4 has all 57 titles accounted for; none are dropped or left only at the parent level. Groups too small to split safely remain a single child.


In [72]:
def choose_child_k(W_local):
    size = len(W_local)
    if size < 3:
        return 1, None, None

    max_k = min(CHILD_K_MAX, size - 1)
    min_k = min(CHILD_K_MIN, max_k)
    if K_CHILD_MODE == "fixed":
        return min(K_CHILD_FIXED, size - 1), None, None

    eigenvalues, eigenvectors = spectral_embedding(W_local, n_eigen=max_k + 1)
    gaps = np.diff(eigenvalues)
    candidate_gaps = gaps[min_k - 1:max_k]
    selected_k = int(np.argmax(candidate_gaps) + min_k)
    return selected_k, eigenvalues, eigenvectors


child_labels = np.empty(n, dtype=int)
child_metadata = {}
next_child_id = 0

for parent_id in sorted(np.unique(parent_labels)):
    parent_indices = np.flatnonzero(parent_labels == parent_id)
    parent_similarity = S[np.ix_(parent_indices, parent_indices)]
    parent_size = len(parent_indices)

    if parent_size < 3:
        local_child_labels = np.zeros(parent_size, dtype=int)
        selected_child_k = 1
    else:
        local_neighbors = min(K_NEIGHBORS, parent_size - 1)
        local_W = build_affinity(parent_similarity, local_neighbors)
        selected_child_k, local_eigenvalues, local_eigenvectors = choose_child_k(local_W)
        if selected_child_k == 1:
            local_child_labels = np.zeros(parent_size, dtype=int)
        else:
            local_child_labels, _ = spectral_cluster(local_eigenvectors, selected_child_k)

    for local_child_id in sorted(np.unique(local_child_labels)):
        child_indices = parent_indices[local_child_labels == local_child_id]
        child_labels[child_indices] = next_child_id
        child_metadata[next_child_id] = {
            "parent_id": int(parent_id),
            "local_child_id": int(local_child_id),
            "size": len(child_indices),
        }
        next_child_id += 1

assert np.all(child_labels >= 0), "every title must receive a child label"
assert len(child_labels) == n
assert sum(info["size"] for info in child_metadata.values()) == n
for parent_id in np.unique(parent_labels):
    parent_size = int((parent_labels == parent_id).sum())
    child_total = sum(
        info["size"] for info in child_metadata.values()
        if info["parent_id"] == int(parent_id)
    )
    assert child_total == parent_size, (
        f"parent {parent_id} has {parent_size} titles but its children contain {child_total}"
    )

child_sizes = Counter(child_labels)
print(f"Child groups: {len(child_sizes)} across {len(parent_sizes)} parents")
print(f"Child sizes (largest first): {[size for _, size in child_sizes.most_common()]}")


Child groups: 18 across 8 parents
Child sizes (largest first): [90, 40, 35, 33, 32, 32, 24, 23, 22, 21, 20, 15, 13, 13, 12, 8, 8, 4]


## 4. Evaluate both levels on the original matrix

These are **internal diagnostics**, not a ground-truth test: the titles have no known correct parent/child labels, and the same similarity matrix is used to build the graph and evaluate the results. They can compare candidate partitions, but cannot prove that a group is a meaningful human category.

- **Silhouette** (higher is better) measures whether items are closer to their own cluster than to the nearest other cluster, using the original precomputed distances.
- **Medoid DB** (lower is better) is a Davies–Bouldin-style score adapted to pairwise distances: it uses each cluster's medoid, not a feature-space centroid. Treat it as a relative diagnostic, not the standard centroid-based DB index.
- **Modularity** (higher is better) measures within-cluster edge weight relative to a weighted-graph null model on the full similarity graph. It often favors coarser partitions and can disagree with silhouette.

The child metrics treat all children as one flat partition; they do not directly score the quality of the parent-to-child hierarchy. Use the actual title lists and cross-parent similarities as important checks.


In [73]:
D = np.clip(1 - S, 0, 1)
np.fill_diagonal(D, 0)
G = S.copy()
np.fill_diagonal(G, 0)


def clustroid(labels):
    result = {}
    for label in np.unique(labels):
        members = np.flatnonzero(labels == label)
        within_distances = D[np.ix_(members, members)]
        result[label] = members[within_distances.sum(axis=1).argmin()]
    return result


def evaluate(labels):
    unique_labels = np.unique(labels)
    count = len(unique_labels)
    metrics = {"silhouette": np.nan, "davies_bouldin": np.nan, "modularity": np.nan}

    if 2 <= count < len(labels):
        metrics["silhouette"] = float(silhouette_score(D, labels, metric="precomputed"))

        centers = clustroid(labels)
        cluster_ids = list(centers)
        spreads = np.array([
            D[labels == cluster_id, centers[cluster_id]].mean()
            for cluster_id in cluster_ids
        ])
        center_indices = [centers[cluster_id] for cluster_id in cluster_ids]
        separation = D[np.ix_(center_indices, center_indices)].copy()
        np.fill_diagonal(separation, np.inf)
        ratios = (spreads[:, None] + spreads[None, :]) / np.maximum(separation, 1e-12)
        metrics["davies_bouldin"] = float(ratios.max(axis=1).mean())

    degree = G.sum(axis=1)
    two_m = degree.sum()
    if two_m > 0:
        same_cluster = labels[:, None] == labels[None, :]
        metrics["modularity"] = float(
            ((G - np.outer(degree, degree) / two_m) * same_cluster).sum() / two_m
        )
    return metrics


parent_metrics = evaluate(parent_labels)
child_metrics = evaluate(child_labels)

print(f"{'Level':<10} {'Groups':>8} {'Silhouette':>12} {'Medoid DB':>10} {'Modularity':>12}")
for name, labels, metrics in [
    ("Parent", parent_labels, parent_metrics),
    ("Child", child_labels, child_metrics),
]:
    print(
        f"{name:<10} {len(np.unique(labels)):>8} "
        f"{metrics['silhouette']:>12.3f} {metrics['davies_bouldin']:>10.3f} "
        f"{metrics['modularity']:>12.3f}"
    )


Level        Groups   Silhouette  Medoid DB   Modularity
Parent            8        0.073      1.747        0.193
Child            18        0.090      1.836        0.151


### Parent-count sweep

For each candidate parent count, this sweep keeps the full affinity graph and its eigenvectors fixed, then runs k-means on the first `k` eigenvectors. It repeats k-means with five seeds and reports mean and standard deviation, so you can see sensitivity to k-means initialization. This does **not** test different graphs, similarity definitions, or data samples.

Interpret the metrics separately: higher silhouette and modularity are generally preferred, while lower Medoid DB is preferred. They can disagree, and silhouette often rises as `k` increases because groups get smaller. The sweep is therefore a sensitivity comparison, **not an automatic selector**. Since the goal is broad parents, choose a count that gives useful group sizes and coherent title lists, even if a finer partition scores better on one metric.


In [74]:
sweep_ks = sorted(set(k for k in PARENT_K_SWEEP if 2 <= k < n) | {K_PARENT})
sweep_rows = []


def mean_and_sd(values):
    values = np.asarray(values, dtype=float)
    return float(values.mean()), float(values.std(ddof=1)) if len(values) > 1 else 0.0


for candidate_k in sweep_ks:
    runs = []
    largest_sizes = []
    smallest_sizes = []
    for sweep_seed in SWEEP_SEEDS:
        candidate_labels, _ = spectral_cluster(parent_eigenvectors, candidate_k, seed=sweep_seed)
        runs.append(evaluate(candidate_labels))
        candidate_sizes = list(Counter(candidate_labels).values())
        largest_sizes.append(max(candidate_sizes))
        smallest_sizes.append(min(candidate_sizes))

    row = {"k": candidate_k}
    for metric in ("silhouette", "davies_bouldin", "modularity"):
        row[f"{metric}_mean"], row[f"{metric}_sd"] = mean_and_sd(
            [run[metric] for run in runs]
        )
    row["largest_mean"] = float(np.mean(largest_sizes))
    row["smallest_mean"] = float(np.mean(smallest_sizes))
    sweep_rows.append(row)

print(f"Repeated k-means seeds: {SWEEP_SEEDS}")
print(
    f"{'k':>3} {'silhouette mean +/- sd':>24} {'Medoid DB mean +/- sd':>24} "
    f"{'modularity mean +/- sd':>25} {'largest':>9} {'smallest':>9}"
)
for row in sweep_rows:
    print(
        f"{row['k']:>3} "
        f"{row['silhouette_mean']:.3f} +/- {row['silhouette_sd']:.3f} "
        f"{row['davies_bouldin_mean']:.3f} +/- {row['davies_bouldin_sd']:.3f} "
        f"{row['modularity_mean']:.3f} +/- {row['modularity_sd']:.3f} "
        f"{row['largest_mean']:>9.1f} {row['smallest_mean']:>9.1f}"
    )

best_silhouette = max(sweep_rows, key=lambda row: row["silhouette_mean"])
best_medoid_db = min(sweep_rows, key=lambda row: row["davies_bouldin_mean"])
best_modularity = max(sweep_rows, key=lambda row: row["modularity_mean"])
print("\nMetric leaders (these do not automatically determine the parent count):")
print(f"  Highest mean silhouette: k={best_silhouette['k']}")
print(f"  Lowest mean Medoid DB:   k={best_medoid_db['k']}")
print(f"  Highest mean modularity: k={best_modularity['k']}")

figure = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Silhouette (higher is better)",
        "Medoid DB (lower is better)",
        "Modularity (higher is better)",
        "Mean cluster sizes",
    ),
    vertical_spacing=0.16,
    horizontal_spacing=0.12,
)

metric_panels = [
    ("silhouette", "silhouette_sd", 1, 1),
    ("davies_bouldin", "davies_bouldin_sd", 1, 2),
    ("modularity", "modularity_sd", 2, 1),
]
for metric, error_metric, row_index, column_index in metric_panels:
    figure.add_trace(
        go.Scatter(
            x=[row["k"] for row in sweep_rows],
            y=[row[f"{metric}_mean"] for row in sweep_rows],
            mode="lines+markers",
            error_y=dict(
                type="data",
                array=[row[error_metric] for row in sweep_rows],
                visible=True,
            ),
            name=metric.replace("_", " "),
            showlegend=False,
        ),
        row=row_index,
        col=column_index,
    )

figure.add_trace(
    go.Scatter(
        x=[row["k"] for row in sweep_rows],
        y=[row["largest_mean"] for row in sweep_rows],
        mode="lines+markers",
        name="Largest cluster",
    ),
    row=2,
    col=2,
)
figure.add_trace(
    go.Scatter(
        x=[row["k"] for row in sweep_rows],
        y=[row["smallest_mean"] for row in sweep_rows],
        mode="lines+markers",
        name="Smallest cluster",
    ),
    row=2,
    col=2,
)
for row_index in (1, 2):
    for column_index in (1, 2):
        figure.add_vline(
            x=K_PARENT,
            line_dash="dash",
            line_color="gray",
            row=row_index,
            col=column_index,
        )
figure.update_xaxes(title_text="Parent count k", row=1, col=1)
figure.update_xaxes(title_text="Parent count k", row=1, col=2)
figure.update_xaxes(title_text="Parent count k", row=2, col=1)
figure.update_xaxes(title_text="Parent count k", row=2, col=2)
figure.update_yaxes(title_text="Score", row=1, col=1)
figure.update_yaxes(title_text="Score", row=1, col=2)
figure.update_yaxes(title_text="Score", row=2, col=1)
figure.update_yaxes(title_text="Titles per cluster", row=2, col=2)
figure.update_layout(
    title="Parent-count sensitivity (error bars show variation across seeds)",
    template="simple_white",
    width=1100,
    height=760,
    showlegend=True,
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))


Repeated k-means seeds: [0, 1, 2, 3, 4]
  k   silhouette mean +/- sd    Medoid DB mean +/- sd    modularity mean +/- sd   largest  smallest
  2 0.046 +/- 0.000 1.859 +/- 0.000 0.181 +/- 0.000     248.0     197.0
  3 0.051 +/- 0.000 1.826 +/- 0.000 0.192 +/- 0.000     201.0      58.0
  4 0.057 +/- 0.000 1.751 +/- 0.000 0.193 +/- 0.000     172.0      54.0
  6 0.064 +/- 0.002 1.707 +/- 0.052 0.197 +/- 0.003     136.6      27.6
  8 0.074 +/- 0.001 1.741 +/- 0.012 0.193 +/- 0.000     131.2      26.2
 10 0.088 +/- 0.000 1.650 +/- 0.001 0.191 +/- 0.000     108.0      14.0
 12 0.093 +/- 0.001 1.662 +/- 0.016 0.178 +/- 0.000      87.4      12.0
 16 0.100 +/- 0.003 1.728 +/- 0.048 0.152 +/- 0.003      50.0      10.4
 20 0.112 +/- 0.002 1.694 +/- 0.052 0.141 +/- 0.003      43.2       8.6
 25 0.124 +/- 0.003 1.655 +/- 0.018 0.126 +/- 0.002      31.2       8.2

Metric leaders (these do not automatically determine the parent count):
  Highest mean silhouette: k=25
  Lowest mean Medoid DB:   k=10
  H

## 5. Inspect which titles belong to each parent and child

The parent itself is a cluster of titles, not a title or a word-based category. Its representative is the medoid: an actual member title with the smallest total distance to other members. Each child heading then lists the titles assigned to that child; all child lists together make up the parent. By default every title is printed (`MAX_TREE_MEMBERS = None`), so you can inspect the complete assignments.


In [75]:
def representative(indices):
    if len(indices) == 1:
        return attributes[indices[0]]
    within_distances = D[np.ix_(indices, indices)]
    return attributes[indices[within_distances.sum(axis=1).argmin()]]


for parent_id in sorted(np.unique(parent_labels)):
    parent_indices = np.flatnonzero(parent_labels == parent_id)
    parent_representative = representative(parent_indices)
    parent_child_ids = [
        child_id for child_id, info in child_metadata.items()
        if info["parent_id"] == int(parent_id)
    ]
    child_counts = [child_metadata[child_id]["size"] for child_id in parent_child_ids]
    child_total = sum(child_counts)
    print(
        f"\nPARENT {parent_id} | {len(parent_indices)} titles | "
        f"representative member: {parent_representative}"
    )
    print(
        f"  All parent titles are assigned to children: "
        f"{' + '.join(map(str, child_counts))} = {child_total}"
    )
    print("  The representative is a member title, not a keyword rule or an extra title.")

    for child_id in parent_child_ids:
        child_indices = np.flatnonzero(child_labels == child_id)
        info = child_metadata[child_id]
        child_representative = representative(child_indices)
        print(
            f"  CHILD {info['local_child_id']} (global id {child_id}) | "
            f"{len(child_indices)} titles | representative member: {child_representative}"
        )

        if MAX_TREE_MEMBERS is None:
            shown = child_indices
        else:
            shown = child_indices[:MAX_TREE_MEMBERS]
        for index in shown:
            print(f"    - {attributes[index]}")
        if len(child_indices) > len(shown):
            print(f"    ... and {len(child_indices) - len(shown)} more")



PARENT 0 | 57 titles | representative member: Insulating
  All parent titles are assigned to children: 33 + 20 + 4 = 57
  The representative is a member title, not a keyword rule or an extra title.
  CHILD 0 (global id 0) | 33 titles | representative member: Insulating
    - Light insulation
    - Insulating
    - Insulating fleece inner
    - Miti Superroubaix high-performance thermal fabric
    - 3mm neoprene
    - Interior waffle-knit pattern
    - Extra height for added insulation
    - Thinsulate™ Insulation
    - Deep winter insulation
    - Insulating fleece inserts
    - Primaloft Gold insulation
    - Thermal knit structure
    - 5 mm neoprene
    - Fleece lining
    - Deep winter protection
    - Cool weather insulation
    - Brushed fleece inserts
    - Insulating neoprene
    - All-season wear
    - Maximum insulation
    - Cold to Mild Weather Performance
    - Miti Lombardia thermal fabric
    - Winter insulation
    - 3 mm neoprene
    - Coated 5 mm neoprene
    - Highl

## 6. t-SNE views of both hierarchy levels

Three views use the **same t-SNE coordinates**. First, the combined view colors by child cluster and marks parent medoids. Second, parent groups are encoded by color and each child index within its parent by marker shape (legend entries show P#/C#). Third, the two-panel view compares parent colors on the left with child colors on the right. Hover any point for its title and both assignments.

t-SNE is for visualization only: it can distort pairwise distances and overlapping points do not prove that the clusters are incorrect. Use the printed hierarchy above to inspect exact membership.


In [76]:
if n < 3:
    raise ValueError("t-SNE visualization needs at least 3 titles")

perplexity = min(30, n - 1)
coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    perplexity=perplexity,
    random_state=SEED,
).fit_transform(D)

parent_colors = qualitative.Dark24
child_colors = qualitative.Alphabet
figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Level 1: parent groups", "Level 2: child groups"),
    horizontal_spacing=0.08,
)

parent_medoids = []
for parent_id in sorted(np.unique(parent_labels)):
    indices = np.flatnonzero(parent_labels == parent_id)
    parent_representative = representative(indices)
    medoid_index = clustroid(parent_labels)[parent_id]
    parent_medoids.append((int(parent_id), medoid_index, parent_representative))
    details = [
        [
            int(parent_id),
            int(child_metadata[int(child_labels[index])]["local_child_id"]),
            int(child_labels[index]),
            parent_representative,
        ]
        for index in indices
    ]
    figure.add_trace(
        go.Scatter(
            x=coordinates[indices, 0],
            y=coordinates[indices, 1],
            mode="markers",
            name=f"Parent {parent_id} ({len(indices)})",
            marker=dict(
                color=parent_colors[int(parent_id) % len(parent_colors)],
                size=8,
                opacity=0.78,
            ),
            text=[attributes[index] for index in indices],
            customdata=details,
            hovertemplate=(
                "<b>%{text}</b><br>Parent %{customdata[0]}: %{customdata[3]}"
                "<br>Child %{customdata[1]} (global id %{customdata[2]})<extra></extra>"
            ),
            showlegend=False,
        ),
        row=1,
        col=1,
    )

for parent_id, medoid_index, parent_representative in parent_medoids:
    figure.add_trace(
        go.Scatter(
            x=[coordinates[medoid_index, 0]],
            y=[coordinates[medoid_index, 1]],
            mode="markers+text",
            marker=dict(symbol="star", size=14, color=parent_colors[int(parent_id) % len(parent_colors)], line=dict(color="white", width=1)),
            text=[f"P{parent_id}"],
            textposition="top center",
            name=f"P{parent_id} medoid",
            customdata=[[parent_representative]],
            hovertemplate="<b>Parent representative</b><br>%{customdata[0]}<extra></extra>",
            showlegend=False,
        ),
        row=1,
        col=1,
    )

for child_id in sorted(child_metadata):
    indices = np.flatnonzero(child_labels == child_id)
    info = child_metadata[child_id]
    parent_id = info["parent_id"]
    parent_indices = np.flatnonzero(parent_labels == parent_id)
    parent_representative = representative(parent_indices)
    child_representative = representative(indices)
    details = [
        [
            parent_id,
            info["local_child_id"],
            parent_representative,
            child_representative,
        ]
        for _ in indices
    ]
    figure.add_trace(
        go.Scatter(
            x=coordinates[indices, 0],
            y=coordinates[indices, 1],
            mode="markers",
            name=(
                f"P{parent_id}/C{info['local_child_id']}: "
                f"{child_representative} ({len(indices)})"
            ),
            marker=dict(
                color=child_colors[child_id % len(child_colors)],
                size=8,
                opacity=0.78,
            ),
            text=[attributes[index] for index in indices],
            customdata=details,
            hovertemplate=(
                "<b>%{text}</b><br>Parent %{customdata[0]}: %{customdata[2]}"
                "<br>Child %{customdata[1]}: %{customdata[3]}<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )

combined_figure = go.Figure()
for child_id in sorted(child_metadata):
    indices = np.flatnonzero(child_labels == child_id)
    info = child_metadata[child_id]
    parent_id = info["parent_id"]
    parent_indices = np.flatnonzero(parent_labels == parent_id)
    parent_representative = representative(parent_indices)
    child_representative = representative(indices)
    details = [
        [
            parent_id,
            info["local_child_id"],
            parent_representative,
            child_representative,
        ]
        for _ in indices
    ]
    combined_figure.add_trace(
        go.Scatter(
            x=coordinates[indices, 0],
            y=coordinates[indices, 1],
            mode="markers",
            name=(
                f"P{parent_id}/C{info['local_child_id']}: "
                f"{child_representative} ({len(indices)})"
            ),
            marker=dict(
                color=child_colors[child_id % len(child_colors)],
                size=8,
                opacity=0.78,
            ),
            text=[attributes[index] for index in indices],
            customdata=details,
            hovertemplate=(
                "<b>%{text}</b><br>Parent %{customdata[0]}: %{customdata[2]}"
                "<br>Child %{customdata[1]}: %{customdata[3]}<extra></extra>"
            ),
        )
    )

for parent_id, medoid_index, parent_representative in parent_medoids:
    combined_figure.add_trace(
        go.Scatter(
            x=[coordinates[medoid_index, 0]],
            y=[coordinates[medoid_index, 1]],
            mode="markers+text",
            marker=dict(
                symbol="star",
                size=14,
                color=parent_colors[int(parent_id) % len(parent_colors)],
                line=dict(color="white", width=1),
            ),
            text=[f"P{parent_id}"],
            textposition="top center",
            name=f"P{parent_id} medoid",
            customdata=[[parent_representative]],
            hovertemplate="<b>Parent representative</b><br>%{customdata[0]}<extra></extra>",
            showlegend=False,
        )
    )

combined_figure.update_xaxes(visible=False)
combined_figure.update_yaxes(visible=False)
combined_figure.update_layout(
    title="Combined hierarchy view: child clusters with parent medoids",
    template="simple_white",
    width=1200,
    height=800,
    legend=dict(
        title="Parent / child representative",
        font=dict(size=9),
        x=1.02,
        y=1,
        itemsizing="constant",
    ),
    hoverlabel=dict(bgcolor="white", font_color="#1f2933"),
)
display(HTML(combined_figure.to_html(full_html=False, include_plotlyjs="cdn")))

hierarchy_figure = go.Figure()
child_shape_symbols = [
    "circle",
    "square",
    "diamond",
    "cross",
    "x",
    "triangle-up",
    "triangle-down",
    "star",
    "pentagon",
    "hexagon",
]
max_local_child_id = max(
    int(info["local_child_id"]) for info in child_metadata.values()
)
if max_local_child_id >= len(child_shape_symbols):
    raise ValueError(
        "The parent-color/child-shape plot supports up to "
        f"{len(child_shape_symbols)} child groups per parent."
    )

for child_id in sorted(child_metadata):
    indices = np.flatnonzero(child_labels == child_id)
    info = child_metadata[child_id]
    parent_id = int(info["parent_id"])
    local_child_id = int(info["local_child_id"])
    parent_indices = np.flatnonzero(parent_labels == parent_id)
    parent_representative = representative(parent_indices)
    child_representative = representative(indices)
    details = [
        [
            parent_id,
            local_child_id,
            parent_representative,
            child_representative,
        ]
        for _ in indices
    ]
    hierarchy_figure.add_trace(
        go.Scatter(
            x=coordinates[indices, 0],
            y=coordinates[indices, 1],
            mode="markers",
            name=f"P{parent_id}/C{local_child_id} ({len(indices)})",
            marker=dict(
                color=parent_colors[parent_id % len(parent_colors)],
                symbol=child_shape_symbols[local_child_id],
                size=9,
                opacity=0.82,
            ),
            text=[attributes[index] for index in indices],
            customdata=details,
            hovertemplate=(
                "<b>%{text}</b><br>Parent %{customdata[0]}: %{customdata[2]}"
                "<br>Child %{customdata[1]}: %{customdata[3]}<extra></extra>"
            ),
        )
    )

hierarchy_figure.update_xaxes(visible=False)
hierarchy_figure.update_yaxes(visible=False)
hierarchy_figure.update_layout(
    title="Hierarchy view: parent group = color; child group = marker shape",
    template="simple_white",
    width=1250,
    height=800,
    legend=dict(
        title="Parent / child (color / shape)",
        font=dict(size=9),
        x=1.02,
        y=1,
        itemsizing="constant",
    ),
    hoverlabel=dict(bgcolor="white", font_color="#1f2933"),
)
display(HTML(hierarchy_figure.to_html(full_html=False, include_plotlyjs="cdn")))

figure.update_xaxes(visible=False)
figure.update_yaxes(visible=False)
figure.update_layout(
    title="Same t-SNE projection, colored at each clustering level",
    template="simple_white",
    width=1500,
    height=800,
    legend=dict(
        title="Parent / child representative",
        font=dict(size=9),
        x=1.02,
        y=1,
        itemsizing="constant",
    ),
    hoverlabel=dict(bgcolor="white", font_color="#1f2933"),
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))


## Alternative approach to consider — NOT implemented in this notebook

Everything above this section uses spectral clustering at both levels on the same blended similarity matrix. The following is a separate possible experiment; none of the code or results above use it.

For that alternative, create broad parent groups by clustering title-plus-description semantic embeddings alone, then use the blended similarity matrix to cluster titles within each parent. Part 3 of `final_report.ipynb` computes semantic vectors and a semantic similarity matrix internally but saves only the blended matrix, so this alternative would require saving or recomputing the semantic output.
